# Driblab - Event Detection: ETL
**Step 1 of the project: load, inspect, and transform the raw data before touching a model.**

This notebook works through the Step 1 checklist from the kickoff guide: the event vocabulary, the events file, the tracking file, and how the two connect. We call this ETL because this step validates and transforms coordinate fields, not just explores them.

**Files used:**
- `dim_event_type.csv` - event type vocabulary (84 defined event types)
- `679026_events.json` - labelled events for match 679026 (Arsenal FC 2-0 FC Brentford)
- `679026_tracking_data.jsonl` - tracking data for the same match

**Structure of this notebook:**
0. Setup - load the data
1. Event Vocabulary - what event types exist vs what actually occurs
2. Events + Tracking - validate event coordinates and normalize tracking coordinates
3. Events File - time encoding, the `outcome` field, and `qualifiers`
4. Tracking File - data quality, visibility, and sampling rate
5. Cross-File Consistency - do events and tracking line up, and what assets do we have?
6. Summary - key takeaways for the team


## Setup — Load Data
**Goal:** load the event-type vocabulary, the labelled events for match 679026, and the corresponding tracking data, so we can explore them below.


In [ ]:
# ── Cell 1: Load all data ─────────────────────────────────────────────────────
import json
import csv
import os
import re
from collections import Counter
import pandas as pd

DATA_DIR = os.path.expanduser('~/Documents/IE/MBDS/Term III/Driblab/data/raw')

# ── Load event type definitions ───────────────────────────────────────────────
DATA_DIR = os.path.expanduser('~/Documents/IE/MBDS/Term III/Driblab/data/raw')
with open(os.path.join(DATA_DIR, 'dim_event_type.csv')) as f:
    event_types = list(csv.DictReader(f))
df_event_types = pd.DataFrame(event_types)

# ── Load labelled events (match 679026) ───────────────────────────────────────
# Note: file has minor corruption, parsing object-by-object
with open(os.path.join(DATA_DIR, '679026_events.json'), errors='ignore') as f:
    content = f.read()
events_raw = []
for match in re.finditer(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', content):
    try:
        events_raw.append(json.loads(match.group()))
    except:
        pass
df_events = pd.json_normalize(events_raw)

# ── Load tracking data (match 679026 — same match as events) ─────────────────
tracking_lines = open(os.path.join(DATA_DIR, '679026_tracking_data.jsonl')).readlines()
tracking_header = json.loads(tracking_lines[0])
tracking_frames = [json.loads(l) for l in tracking_lines[1:]]

print(f"✅ Event types loaded:   {len(df_event_types)} rows")
print(f"✅ Labelled events:      {len(df_events)} rows (679026)")
print(f"✅ Tracking frames:      {len(tracking_frames)} (679026)")
print(f"\nMatch info: {tracking_header['match_data']}")
print(f"\nEvent columns: {list(df_events.columns)}")

**Note:** the events JSON file has minor corruption and is parsed object-by-object (1424 of the events recovered successfully). All three files load correctly and concern the same match (Arsenal FC 2-0 FC Brentford, 2025-12-03).


In [ ]:
tracking_lines = open(os.path.join(DATA_DIR, '679026_tracking_data.jsonl')).readlines()
print(tracking_lines[0])

In [ ]:
# Inspect the tracking JSONL structure
sample_frame = tracking_frames[0]



print("=== Top-level keys in one tracking frame ===")
print(list(sample_frame.keys()))

print("\n=== Ball keys ===")
print(sample_frame['ball'])

print("\n=== Team keys under frame['data'] ===")
print(list(sample_frame['data'].keys()))

print("\n=== Player keys for one player ===")
one_team = next(iter(sample_frame['data'].values()))
one_player = one_team[0]
print(list(one_player.keys()))

print("\n=== Tracking header keys ===")
print(list(tracking_header.keys()))

In [ ]:
# ── Cell 2: Event type distribution across all matches ──────────────────────
from pathlib import Path

data_dir = Path(DATA_DIR) if 'DATA_DIR' in globals() else Path('../data/raw')
event_files = sorted(data_dir.glob('*_events.json'))

all_events_raw = []
for path in event_files:
    with open(path, errors='ignore') as f:
        content = f.read()

    try:
        events = json.loads(content)
    except json.JSONDecodeError:
        events = []
        for match in re.finditer(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', content):
            try:
                events.append(json.loads(match.group()))
            except json.JSONDecodeError:
                pass

    if isinstance(events, dict):
        events = [events]

    all_events_raw.extend(events)

df_all_events = pd.json_normalize(all_events_raw)
event_counts = df_all_events['event.event_type_name'].value_counts()
total = len(df_all_events)

# Build a summary dataframe including percentage and cumulative percentage
df_event_summary = event_counts.rename_axis('event').reset_index(name='count')
df_event_summary['percent'] = df_event_summary['count'] / total * 100
df_event_summary['cumulative_percent'] = df_event_summary['percent'].cumsum()

print(f"Loaded {len(event_files)} event files")
print(f"Total events across all matches: {total:,}")
print(f"Unique event types across all matches: {len(event_counts)} (out of {len(df_event_types)} defined)\n")
# Display the summary table (counts, percent, cumulative_percent)
from IPython.display import display
display(df_event_summary)


**Key insights:**
- Event type coverage is measured across all available event files, not just one match.
- **Severe class imbalance**: PASS dominates the event distribution; use per-class metrics rather than accuracy.
- Rare-but-valuable events (GOAL, SAVED SHOT, MISSED SHOT, CARD) each make up <1% of events — accuracy would be a meaningless metric, per-class F1 is required.


## Part 2 - Events + Tracking: Coordinate System and Attacking Direction
**Goal:** validate that event coordinates are already on the 0-100 attacking-direction scale, then convert tracking ball/player x-y coordinates from physical meters to the same 0-100 field scale for comparison. Step 2 later flips tracking x into the relevant team attacking direction after inferring which team attacks which way.


In [ ]:
# Cell 3: Coordinate system analysis

from IPython.display import display

PITCH_LENGTH_M = 105.0
PITCH_WIDTH_M = 68.0

# Use all loaded event files, not just the displayed match.
events_for_coords = df_all_events.copy() if 'df_all_events' in globals() else df_events.copy()
if 'df_all_events' not in globals():
    print('df_all_events not found; using df_events only. Run Cell 2 first to include every match.')

coord_cols = [c for c in ['x', 'y', 'x_start', 'y_start', 'x_end', 'y_end'] if c in events_for_coords.columns]
events_for_coords[coord_cols] = events_for_coords[coord_cols].apply(pd.to_numeric, errors='coerce')

# Events are already normalized 0-100 by the provider. Do not rescale them here.
df_all_events_checked = events_for_coords.copy()

# Normalize tracking x/y coordinates from meters to the same 0-100 field scale.
tracking_ball_rows = []
tracking_player_rows = []
for frame in tracking_frames:
    ball = frame.get('ball') or [None, None, None]
    clock = frame.get('match_clock') or [None, None]
    ball_x_raw = pd.to_numeric(ball[0], errors='coerce')
    ball_y_raw = pd.to_numeric(ball[1], errors='coerce')
    tracking_ball_rows.append({
        'frame_id': frame.get('frame'),
        'period_id': frame.get('period'),
        'match_clock_min': clock[0],
        'match_clock_sec': clock[1],
        'ball_x_raw_m': ball[0],
        'ball_y_raw_m': ball[1],
        'ball_z_raw_m': ball[2],
        'ball_x_norm': min(100, max(0, ball_x_raw / PITCH_LENGTH_M * 100)) if pd.notna(ball_x_raw) else pd.NA,
        'ball_y_norm': min(100, max(0, ball_y_raw / PITCH_WIDTH_M * 100)) if pd.notna(ball_y_raw) else pd.NA,
    })

    for team_id, players in (frame.get('data') or {}).items():
        for player in players:
            player_x_raw = pd.to_numeric(player.get('x'), errors='coerce')
            player_y_raw = pd.to_numeric(player.get('y'), errors='coerce')
            tracking_player_rows.append({
                'frame_id': frame.get('frame'),
                'period_id': frame.get('period'),
                'match_clock_min': clock[0],
                'match_clock_sec': clock[1],
                'team_id': team_id,
                'player_id': player.get('id'),
                'player_visible': player.get('vis'),
                'player_x_raw_m': player.get('x'),
                'player_y_raw_m': player.get('y'),
                'player_x_norm': min(100, max(0, player_x_raw / PITCH_LENGTH_M * 100)) if pd.notna(player_x_raw) else pd.NA,
                'player_y_norm': min(100, max(0, player_y_raw / PITCH_WIDTH_M * 100)) if pd.notna(player_y_raw) else pd.NA,
            })

df_tracking_ball_normalized = pd.DataFrame(tracking_ball_rows)
df_tracking_players_normalized = pd.DataFrame(tracking_player_rows)

# Events coordinate range across every loaded match.
match_count = df_all_events_checked['match_id'].nunique() if 'match_id' in df_all_events_checked.columns else 1
print('=== EVENTS coordinate system (all matches) ===')
print(f'  Matches: {match_count}')
for col in coord_cols:
    values = df_all_events_checked[col].dropna()
    out_of_range = values.lt(0).sum() + values.gt(100).sum()
    print(f'  {col:<7} range: {values.min()} - {values.max()} | outside 0-100: {out_of_range}')
print('  Events are already normalized 0-100; ETL does not rescale them')

# Do shots cluster at high X? This checks attacking-direction normalization across all matches.
shot_types = ['GOAL', 'SAVED SHOT', 'MISSED SHOT', 'SHOT ON POST']
shots = df_all_events_checked[df_all_events_checked['event.event_type_name'].isin(shot_types)].copy()
shot_summary = (
    shots
    .groupby(['match_id', 'team.team_name', 'period_id'], dropna=False)
    .agg(
        shots=('event.event_type_name', 'size'),
        x_median=('x', 'median'),
        x_min=('x', 'min'),
        x_max=('x', 'max'),
        y_median=('y', 'median'),
    )
    .reset_index()
    .sort_values(['match_id', 'team.team_name', 'period_id'])
)

print()
print('=== SHOT LOCATIONS SUMMARY (all matches, both teams, both halves) ===')
display(shot_summary)

print()
print('=== TRACKING coordinate system (match 679026) ===')
print('Ball coordinates:')
print(df_tracking_ball_normalized[['ball_x_raw_m', 'ball_y_raw_m', 'ball_x_norm', 'ball_y_norm']].describe())
print()
print('Player coordinates:')
print(df_tracking_players_normalized[['player_x_raw_m', 'player_y_raw_m', 'player_x_norm', 'player_y_norm']].describe())

print()
print('Sample normalized tracking ball rows:')
display(df_tracking_ball_normalized.head())
print()
print('Sample normalized tracking player rows:')
display(df_tracking_players_normalized.head())

print()
print('=== SHARED FIELD COORDINATE DECISION ===')
print('  Events:   already 0-100, always attacking toward 100')
print('  Tracking: physical meters on a 105x68 pitch')
print('  ETL:      normalize tracking x/y only, for comparison and modelling')
print('  Step 2:   infer attacking direction and flip event x per half into tracking orientation')


**Key insights:**
- Events already use a **normalized 0-100 coordinate system** for x/y, start x/y, and end x/y across all loaded event files. We validate this; we do not rescale event coordinates.
- Shots cluster at high x across teams, periods, and matches, confirming events are normalized to "always attacking toward x=100", independent of the physical pitch side.
- Tracking starts in **physical meters on a 105x68 pitch**, so ETL converts and clips tracking ball/player x-y into the same **0-100 field scale**.
- Step 2 aligns the two systems by inferring attacking direction and adding tracking-derived attacking-perspective x/y columns. Speeds and distances remain in meters because those are physical features.


## Part 3 — Events File: Time Encoding, Outcome & Qualifiers
**Goal:** check how time is encoded (does the minute reset at half-time?) and compare it to the tracking file's `match_clock`, check what the `outcome` field means across event types, and inspect what's inside the `qualifiers` field.


In [ ]:
# ── Cell 4: Time encoding analysis ───────────────────────────────────────────

# Events time encoding
print("=== EVENTS time encoding ===")
p1 = df_events[df_events['period_id'] == 1]
p2 = df_events[df_events['period_id'] == 2]
print(f"  Period 1 minutes: {p1['min'].min()} - {p1['min'].max()}")
print(f"  Period 2 minutes: {p2['min'].min()} - {p2['min'].max()}")
print(f"  → Minutes count UP and do NOT reset at half-time")

# Tracking time encoding
print(f"\n=== TRACKING time encoding ===")
p1_frames = [f for f in tracking_frames if f['period'] == 1]
p2_frames = [f for f in tracking_frames if f['period'] == 2]
p1_times = [f['match_clock'] for f in p1_frames]
p2_times = [f['match_clock'] for f in p2_frames]
print(f"  Period 1 frames:      {len(p1_frames)}")
print(f"  Period 2 frames:      {len(p2_frames)}")
print(f"  Period 1 match_clock: {p1_times[0]} → {p1_times[-1]}")
print(f"  Period 2 match_clock: {p2_times[0]} → {p2_times[-1]}")

print(f"\n=== TIME SYNC CHALLENGE ===")
print(f"  Events use:   period + min + sec + millisec")
print(f"  Tracking uses: period + match_clock [min, sec] + frame")
print(f"  → match_clock[0]=min, match_clock[1]=sec — compatible!")
print(f"  → But millisec drift may still exist — needs verification")

# ── Outcome field meaning ─────────────────────────────────────────────────────
print(f"\n=== OUTCOME FIELD ===")
outcome_by_type = df_events.groupby('event.event_type_name')['outcome'].unique()
always_true = [name for name, vals in outcome_by_type.items() if set(vals) == {True}]
varies = [name for name, vals in outcome_by_type.items() if len(set(vals)) > 1]
print(f"  Always True (not informative):  {always_true}")
print(f"  Varies True/False (success/fail): {varies}")

# ── Qualifiers structure ──────────────────────────────────────────────────────
print(f"\n=== QUALIFIERS STRUCTURE ===")
sample_quals = next(q for q in df_events['qualifiers'] if isinstance(q, list) and len(q) > 0)
print(f"  Example qualifiers list (one event):")
for q in sample_quals:
    print(f"    {q}")


**Key insights:**
- Event minutes **count up continuously** and do **not** reset at half-time (period 2 starts ~minute 45) — same convention as tracking's `match_clock`, so the two should sync on `[period, min, sec]`. Millisecond-level drift still needs checking once frame-level alignment starts (Step 2).
- `outcome` does **not** mean the same thing for every event type: for BALL TOUCH, BALL RECOVERY and INTERCEPTION it is always `True` (not informative); for PASS, AERIAL, FOUL, TAKEON and TACKLE it varies and represents success/failure.
- `qualifiers` is a list of `{qualifier_id, qualifier_name, value}` dicts with extra event-specific detail (e.g. pass end x/y). It's useful context, not a separate set of events to detect.


## Part 4 — Tracking File: Data Quality & Sampling Rate
**Goal:** check the ball's presence rate, how much player tracking is observed (`vis=True`) vs AI-imputed (`vis=False`), what the `cam` field tells us about reliable live play, and confirm the tracking sampling rate.


In [ ]:
# ── Cell 5: Ball and visibility quality ──────────────────────────────────────

total = len(tracking_frames)

# Ball presence
ball_present = sum(1 for f in tracking_frames if any(v is not None for v in f['ball']))
ball_missing = total - ball_present

# Cam presence (reliable live play)
cam_present = sum(1 for f in tracking_frames if f.get('cam') is not None)

# vis=True vs False
vis_true = vis_false = 0
for f in tracking_frames:
    for team_key in f['data']:
        for p in f['data'][team_key]:
            if p.get('vis'): vis_true += 1
            else: vis_false += 1

# Ball during live play only (cam present)
live_frames = [f for f in tracking_frames if f.get('cam') is not None]
ball_in_live = sum(1 for f in live_frames if any(v is not None for v in f['ball']))

print("=== BALL PRESENCE ===")
print(f"  All frames:        {ball_present:>6} / {total}  ({ball_present/total*100:.1f}%)")
print(f"  Missing:           {ball_missing:>6} / {total}  ({ball_missing/total*100:.1f}%)")
print(f"  During live play:  {ball_in_live:>6} / {cam_present}  ({ball_in_live/cam_present*100:.1f}%)")

print(f"\n=== PLAYER VISIBILITY ===")
total_vis = vis_true + vis_false
print(f"  vis=True  (observed):  {vis_true:>7}  ({vis_true/total_vis*100:.1f}%)")
print(f"  vis=False (imputed):   {vis_false:>7}  ({vis_false/total_vis*100:.1f}%)")

print(f"\n=== CAM FIELD ===")
print(f"  Frames with cam:    {cam_present:>6} / {total}  ({cam_present/total*100:.1f}%)")
print(f"  → Only use cam-present frames for reliable detection")

print(f"\n=== KEY WARNINGS ===")
print(f"  ⚠️  Ball missing in {ball_missing/total*100:.1f}% of all frames")
print(f"  ⚠️  {vis_false/total_vis*100:.1f}% of player positions are AI-imputed, not observed")
print(f"  ⚠️  Ball can be missing exactly during shots and tackles — do not trust blindly")

# ── Sampling rate ──────────────────────────────────────────────────────────────
print(f"\n=== SAMPLING RATE ===")
p1_frames_chk = [f for f in tracking_frames if f['period'] == 1]
start_clock, end_clock = p1_frames_chk[0]['match_clock'], p1_frames_chk[-1]['match_clock']
duration_sec = (end_clock[0]*60 + end_clock[1]) - (start_clock[0]*60 + start_clock[1])
fps = len(p1_frames_chk) / duration_sec
print(f"  Period 1: {len(p1_frames_chk)} frames over ~{duration_sec}s → ~{fps:.1f} Hz")
print(f"  → Confirms ~10 frames/second, as documented")


**Key insights:**
- Tracking runs at **~10 frames/second**, as documented.
- The ball is missing in **47%** of all frames, but only **8.5%** during reliable live play (`cam` present) — still enough to cause issues exactly during fast events like shots and tackles.
- **64%** of player positions are AI-imputed (`vis=False`), not directly observed — treat off-ball positions with some caution.
- Only **58%** of frames have `cam` present — restrict detection logic to these frames for reliability.


## Part 5 — Cross-File Consistency & Data Asset Inventory
**Goal:** verify that team and player IDs match between the events and tracking files for match 679026, sanity-check time alignment, and check which matches in `DATA/` actually have *both* an events file and a tracking file (modelling needs both).


In [ ]:
# ── Cell 6: Cross-file consistency (events ↔ tracking) + asset inventory ────
# Both files are now from match 679026

# ── Data asset inventory ───────────────────────────────────────────────────────
import re as _re
print("=== DATA ASSET INVENTORY ===")
matches = {}
for fn in sorted(os.listdir(DATA_DIR)):
    m = _re.match(r'(\d+)_(events|tracking_data)\.', fn)
    if m:
        matches.setdefault(m.group(1), set()).add(m.group(2))
for match_id, assets in matches.items():
    has_events = 'events' in assets
    has_tracking = 'tracking_data' in assets
    print(f"  Match {match_id}: events={'✅' if has_events else '❌'}  tracking={'✅' if has_tracking else '❌'}")

# Team IDs in tracking
tracking_team_ids = set(tracking_header['players_data'].keys())
print("=== TRACKING team IDs ===")
for team_id in tracking_team_ids:
    players = tracking_header['players_data'][team_id]
    names = [p['name'] for p in players.values()][:3]
    print(f"  team_id: {team_id}  ({len(players)} players)  e.g. {names}")

# Team IDs in events
event_teams = df_events[['team.team_id','team.team_name']].drop_duplicates()
print(f"\n=== EVENTS team IDs ===")
for _, row in event_teams.iterrows():
    print(f"  team_id: {row['team.team_id']}  ({row['team.team_name']})")

# Player ID overlap
tracking_player_ids = set()
for team_id in tracking_header['players_data']:
    for pid in tracking_header['players_data'][team_id].keys():
        tracking_player_ids.add(str(pid))
event_player_ids = set(df_events['player.player_id'].astype(str).unique())
overlap = event_player_ids & tracking_player_ids

print(f"\n=== PLAYER ID OVERLAP ===")
print(f"  Players in tracking: {len(tracking_player_ids)}")
print(f"  Players in events:   {len(event_player_ids)}")
print(f"  Overlap:             {len(overlap)}")
if len(overlap) > 0:
    print(f"  ✅ IDs match — same match confirmed")
else:
    print(f"  ⚠️  No overlap — check if files are from the same match")

# Time alignment sample
print(f"\n=== TIME ALIGNMENT SAMPLE ===")
for _, row in df_events.head(5).iterrows():
    print(f"  Event: {row['event.event_type_name']:<15} P{row['period_id']} {int(row['min']):02d}:{int(row['sec']):02d}  →  tracking match_clock=[{int(row['min'])},{int(row['sec'])}]")

**Key insights:**
- Team IDs and 31/31 player IDs match between the events and tracking files for match 679026 — confirmed same match.
- The first few events line up correctly with tracking's `match_clock`, supporting the time-sync approach from Part 3.
- Of the 3 matches in `DATA/`, only **679026** has both an events file and a tracking file. `678949` has tracking only, and `683231` (FC Barcelona, different league) has events only — so **679026 is currently our only match for the alignment pipeline (Step 2 onward)**.


## Summary - Key Takeaways for the Team
1. **Two coordinate systems**: events are already normalized 0-100 and always attack toward 100; tracking is physical meters on a 105x68 pitch, with teams switching ends at half-time. ETL normalizes tracking x/y only, and Step 2 handles the per-half event flip.
2. **Two time references**: both count up continuously across both halves and look directly compatible on `[period, min, sec]`, but frame-level joining still needs match-clock interpolation at 10 Hz.
3. **The ball is the key signal but incomplete**: present in ~91.5% of live-play frames, never imputed when missing, so gaps need explicit handling.
4. **~64% of player positions are AI-imputed**, not observed, so off-ball features should be treated carefully.
5. **Severe class imbalance**: PASS dominates the event distribution, so use per-class F1 rather than accuracy.
6. **Use all matched event/tracking files for modelling**: Step 2 builds one all-match master join table from every match that has both raw files.

**Next step:** Step 2 - join events to tracking by match clock, filter to `cam`-present frames, interpolate short ball gaps, infer attacking direction, flip event x per half, and build ball/possession/player features.
